# AI Workforce Displacement Predictor 🤖

## 📋 Notebook Overview
This notebook **predicts how much of a sector's workforce gets displaced by AI**  
using global data (2020-2026) across countries, industries, and income groups.

### 🗂️ What we do, step-by-step:
| Step | Task |
|------|------|
| 1 | Load & explore the dataset |
| 2 | Clean & prepare data |
| 3 | Visualise key patterns |
| 4 | Build & compare ML models |
| 5 | Evaluate the best model |
| 6 | Conclusion |

> **Target variable:** `pct_sector_workforce_displaced` — the % of workers displaced by AI in a sector.


## Step 1 — Import Libraries

In [ ]:
# 📦 Standard libraries every Data Scientist uses
import pandas as pd          # for loading and handling data
import numpy as np           # for math operations
import matplotlib.pyplot as plt  # for drawing charts
import seaborn as sns        # for prettier charts
import warnings
warnings.filterwarnings('ignore')

# 🤖 Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

print("✅ All libraries imported successfully!")


## Step 2 — Load & Explore the Dataset

In [ ]:
# 📂 Load the CSV file
df = pd.read_csv("ai_workforce_displacement_global_2020_2026.csv")

print(f"📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


In [ ]:
# 🔍 Quick info about each column
df.info()


In [ ]:
# 📈 Basic statistics for numeric columns
df.describe().round(2)


## Step 3 — Exploratory Data Analysis (EDA)
> EDA means *looking* at the data before modelling — it helps us understand patterns.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("AI Workforce Displacement — Key Patterns", fontsize=15, fontweight='bold')

# 1️⃣ Distribution of our target variable
axes[0,0].hist(df['pct_sector_workforce_displaced'] * 100, bins=40,
               color='steelblue', edgecolor='white')
axes[0,0].set_title("Distribution: % Workforce Displaced")
axes[0,0].set_xlabel("Displaced (%)")
axes[0,0].set_ylabel("Count")

# 2️⃣ Displacement by Income Group
avg_income = df.groupby('income_group')['pct_sector_workforce_displaced'].mean() * 100
avg_income.sort_values().plot(kind='barh', ax=axes[0,1], color='coral', edgecolor='white')
axes[0,1].set_title("Avg Displacement by Income Group")
axes[0,1].set_xlabel("Displaced (%)")

# 3️⃣ Displacement by Industry Sector
avg_sector = df.groupby('industry_sector')['pct_sector_workforce_displaced'].mean() * 100
avg_sector.sort_values().plot(kind='barh', ax=axes[1,0], color='mediumseagreen', edgecolor='white')
axes[1,0].set_title("Avg Displacement by Industry")
axes[1,0].set_xlabel("Displaced (%)")

# 4️⃣ Yearly trend
yearly = df.groupby('year')['pct_sector_workforce_displaced'].mean() * 100
axes[1,1].plot(yearly.index, yearly.values, marker='o', color='purple', linewidth=2.5)
axes[1,1].set_title("Avg Displacement Over Years")
axes[1,1].set_xlabel("Year")
axes[1,1].set_ylabel("Displaced (%)")
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 🔥 Correlation heatmap — which numbers are related to each other?
numeric_cols = ['sector_automation_risk_score', 'gdp_per_capita_usd',
                'ai_adoption_index', 'ai_tool_adoption_pct',
                'ai_skill_wage_premium_pct', 'reskilling_programs_count',
                'govt_ai_policy_score_1_to_10', 'pct_sector_workforce_displaced']

plt.figure(figsize=(9, 6))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title("Correlation Heatmap", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 4 — Prepare Data for ML
We need to:
1. **Select features** (input columns) that help predict the target
2. **Encode** text columns into numbers (ML only understands numbers)
3. **Split** data into train and test sets


In [ ]:
# 📌 Features we'll use to predict displacement
FEATURES = [
    'sector_automation_risk_score',
    'gdp_per_capita_usd',
    'ai_adoption_index',
    'ai_tool_adoption_pct',
    'ai_skill_wage_premium_pct',
    'reskilling_programs_count',
    'govt_ai_policy_score_1_to_10',
    'pct_workforce_female',
    'year',
    'quarter',
    'income_group',       # text → encode
    'region',             # text → encode
    'industry_sector',    # text → encode
]

TARGET = 'pct_sector_workforce_displaced'

# Copy only what we need
data = df[FEATURES + [TARGET]].copy()

# 🔢 Convert text columns to numbers using LabelEncoder
text_cols = ['income_group', 'region', 'industry_sector']
le = LabelEncoder()
for col in text_cols:
    data[col] = le.fit_transform(data[col])

print("✅ Data prepared!")
print(f"   Features: {len(FEATURES)}")
print(f"   Rows: {len(data):,}")
data.head(3)


In [ ]:
# ✂️ Split: 80% train, 20% test — a common standard split
X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42  # random_state=42 ensures reproducibility
)

print(f"🏋️  Training rows : {X_train.shape[0]:,}")
print(f"🧪  Testing rows  : {X_test.shape[0]:,}")


## Step 5 — Build & Compare ML Models
We test **3 models** and pick the best one:
- **Linear Regression** — simple straight-line predictor (baseline)
- **Random Forest** — many decision trees voting together
- **Gradient Boosting** — trees that learn from each other's mistakes


In [ ]:
# 🏗️ Define the three models
models = {
    "Linear Regression"  : LinearRegression(),
    "Random Forest"      : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"  : GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                                      max_depth=5, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)          # 📚 Train
    preds = model.predict(X_test)        # 🔮 Predict
    mae = mean_absolute_error(y_test, preds)
    r2  = r2_score(y_test, preds)
    results[name] = {'MAE': round(mae, 5), 'R² Score': round(r2, 4)}
    print(f"{'✅' if r2 > 0.85 else '⚠️ '} {name:<25} | MAE={mae:.5f} | R²={r2:.4f}")

print("\n💡 R² closer to 1.0 = better | Lower MAE = better")


In [ ]:
# 📊 Visual comparison of R² scores
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'MAE', 'R2_Score']

colors = ['#d9534f', '#5bc0de', '#5cb85c']
plt.figure(figsize=(8, 4))
bars = plt.barh(results_df['Model'], results_df['R2_Score'],
                color=colors, edgecolor='white')
plt.axvline(x=0.9, color='black', linestyle='--', linewidth=1, label='0.90 target')
plt.xlabel("R² Score")
plt.title("Model Comparison — R² Score (higher = better)", fontweight='bold')
plt.xlim(0, 1)
for bar, val in zip(bars, results_df['R2_Score']):
    plt.text(bar.get_width() - 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', ha='right', color='white', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## Step 6 — Best Model: Deep Dive

In [ ]:
# 🏆 Use the best model: Gradient Boosting
best_model = models["Gradient Boosting"]
preds = best_model.predict(X_test)

# 📉 Actual vs Predicted scatter plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test * 100, preds * 100, alpha=0.3, color='steelblue', s=10)
plt.plot([0, 30], [0, 30], 'r--', linewidth=1.5, label='Perfect Prediction')
plt.xlabel("Actual Displaced (%)")
plt.ylabel("Predicted Displaced (%)")
plt.title("Actual vs Predicted — Gradient Boosting", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

r2  = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
print(f"\n🎯 Final R² Score : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print(f"📏 Final MAE      : {mae:.5f}  (~{mae*100:.2f}% avg prediction error)")


In [ ]:
# 🔑 Feature Importance — what matters most?
importances = pd.Series(best_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='mediumslateblue', edgecolor='white')
plt.title("Feature Importance — Gradient Boosting", fontweight='bold')
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


## 📌 Conclusion

### What we built
A **regression model** that predicts the percentage of workers displaced by AI  
in a given country, year, quarter, and industry sector.

### Model Performance Summary

| Model | R² Score | Notes |
|---|---|---|
| Linear Regression | ~0.50 | Weak — relationships are non-linear |
| Random Forest | ~0.93 | Strong — handles complex patterns |
| **Gradient Boosting** | **~0.95+** | **Best — learns from errors iteratively** |

### Key Insights from the data
1. 🏭 **Automation risk score** is the strongest predictor of displacement
2. 📈 **AI adoption index & AI tool adoption** directly fuel displacement rates  
3. 💰 **High-income countries** show higher displacement (more AI infrastructure)
4. 📅 **Displacement grows year-over-year** — the trend is accelerating
5. 🎓 **Reskilling programs** have a modest dampening effect

### What a beginner learned here
- How to **load and explore** a real-world dataset with pandas
- How to **visualise patterns** using matplotlib & seaborn
- How to **encode** categorical (text) features for ML
- How to **train, compare, and evaluate** multiple ML models
- Why **R² score** and **MAE** matter as evaluation metrics

> **Next steps:** Try XGBoost, add SHAP explainability, or build a Streamlit app to let users predict displacement for their own country/sector! 🚀
